# SOXL paper/sim replay production candidate v1

目标：把已经验证的 `schedule -> dashboard -> paper intent` 链路固定成 production 候选 notebook。

边界：v1 只运行 paper/sim，不连接真实 broker，不触发真实下单。生产晋升前仍需更严格模型、风控和外部调度。

In [1]:

from pathlib import Path
import json
import os
import sys

import pandas as pd

REPO_ROOT = Path.cwd().resolve()
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / 'src').exists():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.core import DatasetSnapshot, publish_production_manifest
from src.data import load_soxl_support_frame
from src.dashboard import write_model_matrix_dashboard_export, write_schedule_health
from src.model import QUANTBENCH_MODEL_NAMES
from src.schedule import (
    DataUpdatePipelineConfig,
    ModelDashboardPipelineConfig,
    PaperExecutionPipelineConfig,
    run_data_update_once,
    run_model_dashboard_refresh_once,
    run_paper_execution_once,
)

CARD = Path('production/paper_sim_replay_v1')
CARD.mkdir(parents=True, exist_ok=True)
notifications = []
data_notifications = []
paper_execution_notifications = []
config = ModelDashboardPipelineConfig(
    name='production_paper_sim_replay_v1',
    card_dir=CARD / 'matrix_card',
    dashboard_output_path=CARD / 'dashboard_export.json',
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=tuple(QUANTBENCH_MODEL_NAMES),
    top_n_factors=3,
    dashboard_top_n=5,
    notify_details=True,
    retries=1,
)
market_data_config = DataUpdatePipelineConfig(
    name='production_paper_replay_data_update_v1',
    dataset_key='daily.soxl_paper_replay',
    dataset_name='paper_replay_soxl_daily',
    universe=('SOXL',),
    output_dir=CARD / 'data' / 'snapshots' / 'paper_replay_soxl_daily' / config.end,
    registry_path=CARD / 'data' / 'data_registry.json',
    calendar_path='data/qlib/soxl/calendars/day.txt',
    source_paths=('src.data.load_soxl_support_frame',),
    start=config.start,
    end=config.end,
    as_of=config.end,
    published_at=f'{config.end}T21:00:00Z',
    notify_details=True,
    retries=1,
)
config, market_data_config


(ModelDashboardPipelineConfig(name='production_paper_sim_replay_v1', card_dir=PosixPath('production/paper_sim_replay_v1/matrix_card'), dashboard_output_path=PosixPath('production/paper_sim_replay_v1/dashboard_export.json'), start='2026-01-01', end='2026-06-09', train_end='2026-04-30', model_names=('logistic_clf', 'ridge_ic', 'ridge_mse', 'random_forest_clf', 'hist_gradient_mse', 'xgboost_mse', 'lightgbm_mse', 'catboost_rank', 'extra_trees_rank', 'mlp_mse', 'sequence_lag_ridge_mse', 'lstm_mse', 'gru_mse', 'alstm_mse', 'sfm_mse', 'darnn_mse', 'hawkes_gru_mse', 'tcn_mse', 'mlp_mixer_mse', 'dlinear_mse', 'patch_tst_mse', 'autoformer_mse', 'fedformer_mse', 'market_graph_mse', 'gcn_mse', 'gat_mse', 'rgcn_mse', 'rsr_mse', 'hypergraph_mse', 'thgnn_mse', 'sthan_mse', 'sthgcn_mse'), top_n_factors=3, dashboard_top_n=5, attribution_card_dir=None, production_health_path=None, notebook_card_manifest_path=None, schedule_health_path=None, dashboard_page_output_path=None, dashboard_page_title='SOXL Das

In [2]:
schedule_result = run_model_dashboard_refresh_once(
    config,
    lock_dir=CARD / 'locks',
    notify=notifications.append,
)
schedule_result

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


JobResult(name='production_paper_sim_replay_v1', status='success', payload={'matrix_card': 'production/paper_sim_replay_v1/matrix_card', 'dashboard_export': 'production/paper_sim_replay_v1/dashboard_export.json', 'dashboard_schema_version': 'dashboard.model_matrix.v1', 'dashboard_artifact_id': 'matrix_card', 'dashboard_manifest_kind': 'analysis_manifest', 'dashboard_manifest_path': 'production/paper_sim_replay_v1/matrix_card/analysis_manifest.json', 'dashboard_manifest_hash': '33df91314ce989c110bbb14c268491a84b2a5e998808e06153601d730a326544', 'dashboard_manifest_producer': 'src.analysis.exports', 'dashboard_page': '', 'dashboard_page_schema_version': '', 'dashboard_view_model_schema_version': '', 'dashboard_page_model_card_count': 0, 'top_model': 'mlp_mixer_mse__all_features', 'attribution_card': '', 'attribution_status': 'not_provided', 'matrix_status': 'feature_set_matrix_completed', 'variant_count': 96, 'top_failure_mode': 'return_stability_lag', 'top_failure_segment': 'stress', 'to

In [3]:

dashboard_payload = json.loads((CARD / 'dashboard_export.json').read_text(encoding='utf-8'))
dashboard_payload['top_models'][0]


{'rank': 1,
 'model': 'mlp_mixer_mse__all_features',
 'base_model': 'mlp_mixer_mse',
 'feature_set': 'all_features',
 'return_pct': 55.2857907793225,
 'benchmark_return_pct': 58.118381163943475,
 'relative_return_pct': -2.8325903846209783,
 'max_drawdown_pct': 20.47570313055833,
 'sharpe': 4.121762824528974,
 'avg_position': 0.7634576778205407,
 'turnover': 5.09169811022222,
 'stability_score': 60.7253938909699}

In [4]:

def _paper_replay_frame_provider(_: DataUpdatePipelineConfig) -> pd.DataFrame:
    frame = load_soxl_support_frame(start=config.start, end=config.end).copy()
    frame['instrument'] = 'SOXL'
    return frame.loc[:, ['date', 'instrument', 'open', 'high', 'low', 'close', 'volume', 'soxx_close', 'soxx_volume', 'vix']]


def _paper_replay_calendar_provider(_: DataUpdatePipelineConfig) -> pd.DatetimeIndex:
    frame = load_soxl_support_frame(start=config.start, end=config.end)
    return pd.DatetimeIndex(pd.to_datetime(frame['date']), name='date')


data_update_result = run_data_update_once(
    market_data_config,
    lock_dir=CARD / 'locks',
    notify=data_notifications.append,
    frame_provider=_paper_replay_frame_provider,
    calendar_provider=_paper_replay_calendar_provider,
)
paper_execution_config = PaperExecutionPipelineConfig(
    name='paper_sim_replay_execution_v1',
    dashboard_export_path=CARD / 'dashboard_export.json',
    data_registry_path=market_data_config.registry_path,
    dataset_key=market_data_config.dataset_key,
    instrument='SOXL',
    intent_ledger_path=CARD / 'intent_ledger.json',
    paper_execution_result_path=CARD / 'paper_execution_result.json',
    min_position=0.0,
    max_position=0.75,
    initial_cash=10000.0,
    initial_shares=0,
    notify_details=True,
    retries=1,
)
paper_execution_result = run_paper_execution_once(
    paper_execution_config,
    lock_dir=CARD / 'locks',
    notify=paper_execution_notifications.append,
)
ledger_payload = json.loads((CARD / 'intent_ledger.json').read_text(encoding='utf-8'))
execution_payload = json.loads((CARD / 'paper_execution_result.json').read_text(encoding='utf-8'))
execution_intent_payload = ledger_payload['intent']
market_quote_payload = paper_execution_result.payload['quote']
(CARD / 'execution_intent.json').write_text(json.dumps(execution_intent_payload, ensure_ascii=False, indent=2), encoding='utf-8')
(CARD / 'market_quote.json').write_text(json.dumps(market_quote_payload, ensure_ascii=False, indent=2), encoding='utf-8')
schedule_health_path = CARD / 'schedule_health.json'
schedule_health_payload = write_schedule_health(
    [schedule_result, data_update_result, paper_execution_result],
    output_path=schedule_health_path,
)
dashboard_payload = write_model_matrix_dashboard_export(
    CARD / 'matrix_card',
    output_path=CARD / 'dashboard_export.json',
    top_n=config.dashboard_top_n,
    schedule_health_path=schedule_health_path,
)
{
    'data_update': data_update_result.payload,
    'paper_execution': paper_execution_result.payload,
    'schedule_health': schedule_health_payload,
    'market_quote': market_quote_payload,
    'execution_result': execution_payload,
}


{'data_update': {'schema_version': 'schedule.data_update_result.v1',
  'dataset_key': 'daily.soxl_paper_replay',
  'dataset_name': 'paper_replay_soxl_daily',
  'status': 'published',
  'manifest_path': 'production/paper_sim_replay_v1/data/snapshots/paper_replay_soxl_daily/2026-06-09/data_update_manifest.json',
  'manifest_hash': 'c2d66c782be2de4adb47554cd4c5211481a9be3c110bc90fdc7da6f855a317a7',
  'dataset_schema_hash': '956800a74aa0ed21477e2ffba40a4359641ee6bcfb4347f15d6eae3ee118b5a3',
  'registry_path': 'production/paper_sim_replay_v1/data/data_registry.json',
  'start': '2026-01-02',
  'end': '2026-06-09',
  'as_of': '2026-06-09',
  'row_count': 109,
  'column_count': 10,
  'frame_schema_hash': 'cc88b355f6e3340e9bca290d2bc329530b03ecafc6a5f78eb9be4daee9b08f71',
  'dropped_non_calendar_rows': 0,
  'published_at': '2026-06-09T21:00:00Z',
  'attempts': 1,
  'retry_schema_version': 'schedule.retry_attempts.v1',
  'retry_error_count': 0,
  'retry_errors': []},
 'paper_execution': {'schem

In [5]:

intent_metadata = dict(execution_intent_payload.get('metadata') or {})
analysis_overview = dashboard_payload.get('analysis_overview', {})
top_gap = (analysis_overview.get('largest_relative_return_spread_segments') or [{}])[0]
model_difference_explanations = dashboard_payload.get('model_difference_explanations')
if not isinstance(model_difference_explanations, list):
    model_difference_explanations = []
top_model_difference = model_difference_explanations[0] if model_difference_explanations else {}
market_quote_payload = dict(paper_execution_result.payload.get('quote') or {})
snapshot_payload = dict(paper_execution_result.payload.get('snapshot') or {})
summary = {
    'schema_version': 'production.paper_sim_replay.v1',
    'status': 'paper_replay_completed',
    'mode': execution_intent_payload.get('mode', ''),
    'live_broker_enabled': False,
    'instrument': execution_intent_payload.get('instrument', ''),
    'source_model': execution_intent_payload.get('source_model', ''),
    'target_position': execution_intent_payload.get('target_position', 0.0),
    'execution_gate': intent_metadata.get('execution_gate', ''),
    'promotion_decision': intent_metadata.get('promotion_decision', ''),
    'promotion_blockers': intent_metadata.get('promotion_blockers', ''),
    'analysis_overview_schema': intent_metadata.get('analysis_overview_schema_version', ''),
    'analysis_top_gap_segment': intent_metadata.get('analysis_top_gap_segment', ''),
    'analysis_top_gap_spread': intent_metadata.get('analysis_top_gap_spread', 0.0),
    'analysis_top_gap_best_model': intent_metadata.get('analysis_top_gap_best_model', ''),
    'analysis_top_gap_worst_model': intent_metadata.get('analysis_top_gap_worst_model', ''),
    'analysis_top_weakness_model': intent_metadata.get('analysis_top_weakness_model', ''),
    'analysis_top_weakness_decision_hint': intent_metadata.get('analysis_top_weakness_decision_hint', ''),
    'model_difference_schema_version': top_model_difference.get('schema_version', ''),
    'model_difference_count': len(model_difference_explanations),
    'model_difference_top_signal': top_model_difference.get('explanation_signal', ''),
    'model_difference_top_decision_hint': top_model_difference.get('decision_hint', ''),
    'model_difference_top_segment': top_model_difference.get('segment', ''),
    'model_difference_top_winner_model': top_model_difference.get('winner_model', ''),
    'model_difference_top_loser_model': top_model_difference.get('loser_model', ''),
    'model_difference_top_spread': top_model_difference.get('relative_return_pct_spread', 0.0),
    'backtest_schema_version': intent_metadata.get('backtest_schema_version', ''),
    'requested_model_schema_version': intent_metadata.get('requested_model_schema_version', ''),
    'requested_model_count': intent_metadata.get('requested_model_count', 0),
    'requested_model_completed_count': intent_metadata.get('requested_model_completed_count', 0),
    'requested_model_all_completed': intent_metadata.get('requested_model_all_completed', False),
    'requested_model_missing_count': intent_metadata.get('requested_model_missing_count', 0),
    'requested_model_incomplete_count': intent_metadata.get('requested_model_incomplete_count', 0),
    'requested_model_missing_models': intent_metadata.get('requested_model_missing_models', ''),
    'backtest_walk_forward_fold_count': intent_metadata.get('backtest_walk_forward_fold_count', 0),
    'backtest_slippage_bps': intent_metadata.get('backtest_slippage_bps', 0.0),
    'market_data_adapter': 'paper_market_data',
    'market_data_dataset_key': market_data_config.dataset_key,
    'market_data_registry_path': str(market_data_config.registry_path),
    'market_data_manifest_path': data_update_result.payload.get('manifest_path', ''),
    'market_data_manifest_hash': data_update_result.payload.get('manifest_hash', ''),
    'market_quote': market_quote_payload,
    'paper_shares': snapshot_payload.get('shares', 0),
    'paper_position': snapshot_payload.get('position', 0.0),
    'paper_execution_pipeline_schema_version': paper_execution_result.payload.get('schema_version', ''),
    'paper_execution_job_status': paper_execution_result.status,
    'paper_execution_attempts': paper_execution_result.payload.get('attempts'),
    'paper_execution_result': str(CARD / 'paper_execution_result.json'),
    'schedule_health': str(schedule_health_path),
    'schedule_health_schema_version': schedule_health_payload.get('schema_version', ''),
    'schedule_health_job_count': schedule_health_payload.get('job_count', 0),
    'schedule_health_failed_job_count': schedule_health_payload.get('failed_job_count', 0),
    'schedule_health_total_retry_error_count': schedule_health_payload.get('total_retry_error_count', 0),
    'schedule_health_top_failure_job': schedule_health_payload.get('top_failure_job', ''),
    'schedule_health_top_failure_last_error': schedule_health_payload.get('top_failure_last_error', ''),
    'schedule_status': schedule_result.status,
    'schedule_attempts': schedule_result.payload.get('attempts'),
    'schedule_readiness_overview_schema': schedule_result.payload.get('readiness_overview_schema_version', ''),
    'schedule_readiness_overview_status': schedule_result.payload.get('readiness_overview_status', ''),
    'schedule_readiness_overview_gate_count': schedule_result.payload.get('readiness_overview_gate_count', 0),
    'schedule_readiness_overview_pass_count': schedule_result.payload.get('readiness_overview_pass_count', 0),
    'schedule_readiness_overview_review_count': schedule_result.payload.get('readiness_overview_review_count', 0),
    'schedule_readiness_overview_block_count': schedule_result.payload.get('readiness_overview_block_count', 0),
    'schedule_readiness_overview_gates': schedule_result.payload.get('readiness_overview_gates', ''),
    'schedule_readiness_overview_review_gates': schedule_result.payload.get('readiness_overview_review_gates', ''),
    'schedule_readiness_overview_block_gates': schedule_result.payload.get('readiness_overview_block_gates', ''),
    'data_update_status': data_update_result.status,
    'data_update_attempts': data_update_result.payload.get('attempts'),
    'data_update_row_count': data_update_result.payload.get('row_count'),
    'data_update_as_of': data_update_result.payload.get('as_of'),
    'notifications': notifications,
    'data_notifications': data_notifications,
    'paper_execution_notifications': paper_execution_notifications,
    'dashboard_export': str(CARD / 'dashboard_export.json'),
    'intent_ledger': str(CARD / 'intent_ledger.json'),
    'matrix_card': str(CARD / 'matrix_card'),
    'guardrail': execution_intent_payload.get('guardrail', ''),
    'analysis_overview': analysis_overview,
    'top_gap': top_gap,
}
(CARD / 'production_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

production_dataset = DatasetSnapshot(
    name=market_data_config.dataset_name,
    universe=market_data_config.universe,
    start=data_update_result.payload.get('start', config.start),
    end=data_update_result.payload.get('end', config.end),
    calendar_path=market_data_config.calendar_path,
    data_paths=(str(market_data_config.registry_path), data_update_result.payload.get('manifest_path', '')),
    metadata={
        'market_data_dataset_key': market_data_config.dataset_key,
        'market_data_manifest_hash': data_update_result.payload.get('manifest_hash', ''),
        'dashboard_manifest_hash': schedule_result.payload.get('dashboard_manifest_hash', ''),
        'dashboard_manifest_kind': schedule_result.payload.get('dashboard_manifest_kind', ''),
        'paper_execution_pipeline_schema_version': summary['paper_execution_pipeline_schema_version'],
        'schedule_health_schema_version': summary['schedule_health_schema_version'],
        'schedule_health_job_count': summary['schedule_health_job_count'],
        'schedule_health_failed_job_count': summary['schedule_health_failed_job_count'],
    },
)
production_manifest = publish_production_manifest(
    strategy_id='paper_sim_replay_v1',
    output_dir=CARD,
    registry_path=Path('production') / 'production_registry.json',
    notebook_path='paper_sim_replay_v1.ipynb',
    dataset=production_dataset,
    paths={
        'summary': 'production_summary.json',
        'dashboard_export': 'dashboard_export.json',
        'schedule_health': 'schedule_health.json',
        'intent_ledger': 'intent_ledger.json',
        'execution_intent': 'execution_intent.json',
        'market_quote': 'market_quote.json',
        'paper_execution_result': 'paper_execution_result.json',
        'data_registry': 'data/data_registry.json',
        'data_update_manifest': str(Path(data_update_result.payload.get('manifest_path', '')).relative_to(CARD)),
        'analysis_manifest': 'matrix_card/analysis_manifest.json',
        'model_artifact_manifest': 'matrix_card/artifact_manifest.json',
    },
    metadata={
        'status': summary['status'],
        'mode': summary['mode'],
        'live_broker_enabled': summary['live_broker_enabled'],
        'instrument': summary['instrument'],
        'source_model': summary['source_model'],
        'target_position': summary['target_position'],
        'execution_gate': summary['execution_gate'],
        'promotion_decision': summary['promotion_decision'],
        'market_data_dataset_key': summary['market_data_dataset_key'],
        'market_quote_date': summary['market_quote']['date'],
        'market_quote_price': summary['market_quote']['price'],
        'paper_position': summary['paper_position'],
        'paper_shares': summary['paper_shares'],
        'schedule_status': summary['schedule_status'],
        'schedule_health_schema_version': summary['schedule_health_schema_version'],
        'schedule_health_job_count': summary['schedule_health_job_count'],
        'schedule_health_failed_job_count': summary['schedule_health_failed_job_count'],
        'schedule_health_total_retry_error_count': summary['schedule_health_total_retry_error_count'],
        'schedule_readiness_overview_schema': summary['schedule_readiness_overview_schema'],
        'schedule_readiness_overview_status': summary['schedule_readiness_overview_status'],
        'schedule_readiness_overview_gates': summary['schedule_readiness_overview_gates'],
        'data_update_status': summary['data_update_status'],
        'paper_execution_pipeline_schema_version': summary['paper_execution_pipeline_schema_version'],
        'paper_execution_job_status': summary['paper_execution_job_status'],
        'analysis_overview_schema': summary['analysis_overview_schema'],
        'backtest_schema_version': summary['backtest_schema_version'],
    },
    published_at=f'{config.end}T21:30:00Z',
)
summary['production_manifest_path'] = str(CARD / 'production_manifest.json')
summary['production_manifest_hash'] = production_manifest['manifest_hash']
summary['production_registry_path'] = 'production/production_registry.json'
(CARD / 'production_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary


{'schema_version': 'production.paper_sim_replay.v1',
 'status': 'paper_replay_completed',
 'mode': 'paper',
 'live_broker_enabled': False,
 'instrument': 'SOXL',
 'source_model': 'mlp_mixer_mse__all_features',
 'target_position': 0.75,
 'execution_gate': 'paper_review',
 'promotion_decision': 'promote_candidate',
 'promotion_blockers': '',
 'analysis_overview_schema': 'analysis.overview.v1',
 'analysis_top_gap_segment': 'downtrend',
 'analysis_top_gap_spread': 72.55670639330125,
 'analysis_top_gap_best_model': 'sfm_mse__all_features',
 'analysis_top_gap_worst_model': 'ridge_ic__all_features',
 'analysis_top_weakness_model': 'fedformer_mse__top_ic_factors',
 'analysis_top_weakness_decision_hint': 'block_or_rework',
 'model_difference_schema_version': 'analysis.model_difference_explanation.v1',
 'model_difference_count': 34,
 'model_difference_top_signal': 'segment_winner',
 'model_difference_top_decision_hint': 'promote_segment_candidate',
 'model_difference_top_segment': 'downtrend',
 

In [6]:

assert summary['execution_gate'] in {'paper_review', 'blocked_to_min_position'}
assert summary['analysis_overview_schema'] == 'analysis.overview.v1'
assert summary['promotion_decision']
assert summary['backtest_schema_version'] == 'backtest.config.v1'
assert summary['requested_model_schema_version'] == 'model.requested_contract_health.v1'
assert summary['requested_model_count'] == len(QUANTBENCH_MODEL_NAMES)
assert summary['requested_model_completed_count'] == summary['requested_model_count']
assert summary['requested_model_all_completed'] is True
assert summary['requested_model_missing_count'] == 0
assert summary['model_difference_schema_version'] == 'analysis.model_difference_explanation.v1'
assert summary['model_difference_count'] >= 1
assert summary['model_difference_top_signal']
assert summary['model_difference_top_decision_hint']
assert summary['backtest_walk_forward_fold_count'] >= 1
assert summary['market_data_dataset_key'] == 'daily.soxl_paper_replay'
assert summary['data_update_status'] == 'success'
assert summary['paper_execution_pipeline_schema_version'] == 'schedule.paper_execution.v1'
assert summary['paper_execution_job_status'] == 'success'
assert summary['paper_execution_result'] == str(CARD / 'paper_execution_result.json')
assert summary['production_manifest_hash'] == production_manifest['manifest_hash']
assert production_manifest['schema_version'] == 'production.manifest.v1'
assert production_manifest['paths']['intent_ledger'] == 'intent_ledger.json'
assert production_manifest['paths']['data_registry'] == 'data/data_registry.json'
assert production_manifest['paths']['schedule_health'] == 'schedule_health.json'
assert summary['market_quote']['source_path'].endswith('canonical_frame.csv')
assert 'data/snapshots/' in summary['market_quote']['source_path']
assert 'regime_gap=' in ' '.join(summary['notifications'])
assert summary['schedule_readiness_overview_schema'] == 'dashboard.readiness_overview.v1'
assert summary['schedule_readiness_overview_gate_count'] >= 1
assert summary['schedule_readiness_overview_block_count'] == 0
assert summary['schedule_health_schema_version'] == 'dashboard.schedule_health.v1'
assert summary['schedule_health_job_count'] == 3
assert summary['schedule_health_failed_job_count'] == 0
assert summary['schedule_health_total_retry_error_count'] == 0
assert production_manifest['metadata']['schedule_health_schema_version'] == summary['schedule_health_schema_version']
assert 'readiness=' in ' '.join(summary['notifications'])
assert 'dataset=daily.soxl_paper_replay' in ' '.join(summary['data_notifications'])
assert 'paper_sim_replay_execution_v1:success' in ' '.join(summary['paper_execution_notifications'])
assert intent_metadata['execution_gate'] == summary['execution_gate']

{
    'dashboard_top_models': dashboard_payload['top_models'][:3],
    'feature_set_summary': dashboard_payload['feature_set_summary'],
    'execution_gate': summary['execution_gate'],
    'promotion_decision': summary['promotion_decision'],
    'analysis_top_gap_segment': summary['analysis_top_gap_segment'],
    'model_difference_count': summary['model_difference_count'],
    'model_difference_top_signal': summary['model_difference_top_signal'],
    'requested_model_completed_count': summary['requested_model_completed_count'],
    'requested_model_count': summary['requested_model_count'],
    'backtest_schema_version': summary['backtest_schema_version'],
    'market_data_dataset_key': summary['market_data_dataset_key'],
    'market_quote_source_path': summary['market_quote']['source_path'],
    'paper_execution_pipeline_schema_version': summary['paper_execution_pipeline_schema_version'],
    'schedule_readiness_overview_status': summary['schedule_readiness_overview_status'],
    'schedule_readiness_overview_gates': summary['schedule_readiness_overview_gates'],
    'schedule_health_schema_version': summary['schedule_health_schema_version'],
    'schedule_health_job_count': summary['schedule_health_job_count'],
    'schedule_health_failed_job_count': summary['schedule_health_failed_job_count'],
    'production_manifest_hash': summary['production_manifest_hash'],
    'ledger_exists': (CARD / 'intent_ledger.json').exists(),
    'live_broker_enabled': summary['live_broker_enabled'],
}


{'dashboard_top_models': [{'rank': 1,
   'model': 'mlp_mixer_mse__all_features',
   'base_model': 'mlp_mixer_mse',
   'feature_set': 'all_features',
   'return_pct': 55.2857907793225,
   'benchmark_return_pct': 58.118381163943475,
   'relative_return_pct': -2.8325903846209783,
   'max_drawdown_pct': 20.47570313055833,
   'sharpe': 4.121762824528974,
   'avg_position': 0.7634576778205407,
   'turnover': 5.09169811022222,
   'stability_score': 60.7253938909699},
  {'rank': 2,
   'model': 'mlp_mixer_mse__ic_weighted_composite',
   'base_model': 'mlp_mixer_mse',
   'feature_set': 'ic_weighted_composite',
   'return_pct': 45.75122150395392,
   'benchmark_return_pct': 58.118381163943475,
   'relative_return_pct': -12.367159659989554,
   'max_drawdown_pct': 26.56330283032109,
   'sharpe': 3.554521316797977,
   'avg_position': 0.7403846153846154,
   'turnover': 0.25,
   'stability_score': 47.592306134032334},
  {'rank': 3,
   'model': 'rgcn_mse__top_ic_factors',
   'base_model': 'rgcn_mse',
  


## 中文结论

- 做了什么：运行 production 候选链路，完成 model/dashboard schedule 刷新、data update schedule 发布 canonical snapshot 与 data registry，并通过 `src.schedule.run_paper_execution_once` 串联 dashboard export、data registry、paper market quote、execution intent、PaperBrokerAdapter、intent ledger 和 paper execution result；随后把 dashboard refresh、data update、paper execution 三段实际 JobResult 汇总为 `schedule_health.json`，最后发布 `production_manifest.json` 与 `production_registry.json`。
- 核心指标：source model、target_position、market quote、paper shares、paper position、execution_gate、promotion_decision、backtest schema、requested_models=32/32、market_data_dataset_key、data_update_status、paper_execution_pipeline_schema_version、schedule readiness overview、schedule_health job_count/failed_job_count/retry_error_count、model_difference_explanations、production_manifest_hash 均写入 `production_summary.json` 与相关 sidecar。
- regime 差异：dashboard export 保留 `analysis_overview.v1` 和 `analysis.model_difference_explanation.v1`，paper intent metadata 记录最大 `regime_gap`、top weakness、模型差异解释和 promotion 决策，生产回放不再只看全区间仓位。
- 是否晋升：否。本 notebook 是 paper/sim production candidate，不是 live 生产策略；但 paper 执行路径已从手写 API helper 晋升为 schedule 层稳定管线。
- 风险：`live_broker_enabled=False`；guardrail=`v1 只允许 paper/sim，不连接真实 broker，不触发真实下单。`；quote 来自 as-of data registry current snapshot，schedule readiness overview 汇总 dashboard export 中已有 gate，`schedule_health.json` 汇总本 notebook 内三段 schedule 运行健康状态，仍不能替代外部调度器审计、跨日锁检查和 live adapter 权限隔离。
- 下一步：把 `schedule_health.json` 接入 production health 通知或 operator dashboard 的入口检查，并在不改变 v1 paper/sim 边界的前提下补 scheduler 入口文档。
